In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob

# -----------------------------------------------
# 1. LOAD AND COMBINE ALL GW FILES
# -----------------------------------------------
# Only reads gw*.csv files - avoids accidentally picking up players.csv or player_scores.csv

folder = r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload"
all_files = glob(os.path.join(folder, "gw*.csv"))

print(f"Found {len(all_files)} GW files:")
for f in sorted(all_files):
    print(f"  {os.path.basename(f)}")

if len(all_files) == 0:
    raise FileNotFoundError(f"No gw*.csv files found in {folder}. Run gameweek_pull.ipynb first.")

df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
print(f"Total rows loaded: {len(df)}")
print(df.columns.tolist())

Found 33 GW files:
  gw1.csv
  gw10.csv
  gw11.csv
  gw12.csv
  gw13.csv
  gw14.csv
  gw15.csv
  gw16.csv
  gw17.csv
  gw18.csv
  gw19.csv
  gw2.csv
  gw20.csv
  gw21.csv
  gw22.csv
  gw23.csv
  gw24.csv
  gw25.csv
  gw26.csv
  gw27.csv
  gw28.csv
  gw29.csv
  gw3.csv
  gw30.csv
  gw31.csv
  gw32.csv
  gw33.csv
  gw4.csv
  gw5.csv
  gw6.csv
  gw7.csv
  gw8.csv
  gw9.csv
Total rows loaded: 25893
['player_id', 'code', 'gameweek', 'fixture_id', 'name', 'web_name', 'element_type', 'selected_by_percent', 'value', 'team_id', 'opponent_id', 'h_a', 'kickoff_time', 'team_score', 'opponent_score', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'yellow_cards', 'red_cards', 'bonus', 'bps', 'total_points', 'xG', 'xA', 'xGI', 'xGC', 'DefCon', 'ict_index', 'influence', 'creativity', 'threat', 'in_dreamteam', 'status', 'news', 'corners_indirect_freekicks_order', 'direct_freekicks_order', 'penalties_order']


In [2]:
# -----------------------------------------------
# 2. FILTER TO PLAYERS WITH MINUTES > 0
# -----------------------------------------------
# Removes players who never played - stops them distorting percentiles

df = df[df["minutes"] > 0].copy()

In [3]:
# -----------------------------------------------
# 3. MINUTES SCORE (max-based, not percentile)
# -----------------------------------------------
# Max possible mins = highest gameweek number * 90
# Each player gets a % of that, scaled to 1-10

max_gw = df["gameweek"].max()
max_possible_mins = max_gw * 90

# NOTE: grouping by 'code' (original API name) — matches Power BI column name
df = df.sort_values(["code", "gameweek"])
df["cumulative_mins"] = df.groupby("code")["minutes"].cumsum()

df["mins_score"] = (df["cumulative_mins"] / max_possible_mins * 9 + 1).clip(1, 10)

In [4]:
# -----------------------------------------------
# 4. PERCENTILE SCORES (within position, cumulative)
# -----------------------------------------------
# For each metric:
#   a) Calculate each player's cumulative total up to that GW
#   b) Rank them within their position at that GW snapshot
#   c) Convert rank to a 1-10 score

# Metrics where MORE = BETTER
positive_metrics = {
    "goals_scored": "goals_score",
    "assists":      "assists_score",
    "xG":           "xg_score",
    "xGI":          "xgi_score",
    "clean_sheets": "cs_score",
    "bonus":        "bonus_score",
}

# Metrics where LESS = BETTER (we invert the percentile)
negative_metrics = {
    "goals_conceded": "goals_conceded_score",
    "xGC":            "xgc_score",
}

def percentile_score(series, invert=False):
    """
    Takes a series of cumulative values.
    Returns a 1-10 score based on percentile rank.
    If invert=True, lower values get higher scores.
    """
    pct = series.rank(pct=True)
    if invert:
        pct = 1 - pct
    return (pct * 9 + 1).clip(1, 10)

# Calculate cumulative totals and percentile scores for each metric
# NOTE: grouping by 'code' and 'element_type' — original API names matching Power BI
for col, score_col in {**positive_metrics, **negative_metrics}.items():
    invert = col in negative_metrics

    cum_col = f"cum_{col}"
    df[cum_col] = df.groupby("code")[col].cumsum()

    df[score_col] = (
        df.groupby(["gameweek", "element_type"])[cum_col]
        .transform(lambda x: percentile_score(x, invert=invert))
    )

In [5]:
# -----------------------------------------------
# 5. COMPOSITE SCORE (weighted average)
# -----------------------------------------------
# All weights add up to 1

weights = {
    "mins_score":           0.10,
    "goals_score":          0.20,
    "assists_score":        0.10,
    "xg_score":             0.10,
    "xgi_score":            0.25,
    "cs_score":             0.10,
    "bonus_score":          0.05,
    "goals_conceded_score": 0.05,
    "xgc_score":            0.05,
}

df["composite_score"] = sum(
    df[score] * weight for score, weight in weights.items()
)

In [6]:
# -----------------------------------------------
# 6. SELECT OUTPUT COLUMNS
# -----------------------------------------------
# NOTE: using 'code' and 'element_type' — original API names matching Power BI

output_cols = [
    "code",
    "web_name",
    "element_type",
    "gameweek",
    "minutes",
    "mins_score",
    "goals_score",
    "assists_score",
    "xg_score",
    "xgi_score",
    "cs_score",
    "bonus_score",
    "goals_conceded_score",
    "xgc_score",
    "composite_score",
]

output = df[output_cols].sort_values(["gameweek", "composite_score"], ascending=[True, False])

In [7]:
output.head(15)

,code,web_name,element_type,gameweek,minutes,mins_score,goals_score,assists_score,xg_score,xgi_score,cs_score,bonus_score,goals_conceded_score,xgc_score,composite_score
429,223094,Haaland,4,1,72,1.218182,9.718750,5.500000,10.000000,10.000000,9.156250,9.578125,7.187500,5.781250,8.158537
6,466075,Calafiori,2,1,71,1.215152,9.955000,5.455000,10.000000,10.000000,8.425000,9.505000,7.750000,5.815000,8.154015
163,219249,O'Riley,3,1,87,1.263636,9.724490,5.010204,9.938776,9.969388,8.530612,9.908163,7.336735,6.387755,8.093200
530,223827,Ballard,2,1,90,1.272727,9.955000,5.455000,9.460000,8.920000,8.425000,9.865000,7.750000,7.570000,7.941523
426,433036,Reijnders,3,1,90,1.272727,9.724490,9.479592,8.806122,8.591837,8.530612,5.316327,7.336735,5.867347,7.827783
579,242898,Johnson,3,1,79,1.239394,9.724490,5.010204,9.755102,9.326531,8.530612,5.316327,7.336735,4.520408,7.588735
596,212319,Richarlison,4,1,71,1.215152,9.718750,5.500000,8.593750,8.593750,9.156250,9.578125,7.187500,3.812500,7.567609
660,510663,Ekitiké,4,1,71,1.215152,8.453125,10.000000,9.718750,9.718750,4.656250,8.734375,3.671875,2.968750,7.448078
346,243526,Gudmundsson,2,1,90,1.272727,5.455000,5.455000,9.235000,9.730000,8.425000,9.865000,7.750000,7.975000,7.241773
581,460842,Kudus,3,1,84,1.254545,5.255102,10.000000,6.877551,9.602041,8.530612,9.724490,7.336735,4.520408,7.196883


In [8]:
# -----------------------------------------------
# 7. SAVE OUTPUT
# -----------------------------------------------

output["code"] = output["code"].astype(int)
output["element_type"] = output["element_type"].astype(int)
output["gameweek"] = output["gameweek"].astype(int)
output["minutes"] = output["minutes"].astype(int)

output_path = r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload\player_scores.csv"
output.to_csv(output_path, index=False)

print(f"Done! {len(output)} rows saved to {output_path}")
print(output.head(10))

Done! 10040 rows saved to B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload\player_scores.csv
       code     web_name  element_type  gameweek  minutes  mins_score  \
429  223094      Haaland             4         1       72    1.218182   
6    466075    Calafiori             2         1       71    1.215152   
163  219249      O'Riley             3         1       87    1.263636   
530  223827      Ballard             2         1       90    1.272727   
426  433036    Reijnders             3         1       90    1.272727   
579  242898      Johnson             3         1       79    1.239394   
596  212319  Richarlison             4         1       71    1.215152   
660  510663      Ekitiké             4         1       71    1.215152   
346  243526  Gudmundsson             2         1       90    1.272727   
581  460842        Kudus             3         1       84    1.254545   

     goals_score  assists_score   xg_score  xgi_score  cs_score  bonus_score  \
429    

In [9]:
# Quick check of output schema
print(output.dtypes)

code                      int64
web_name                 object
element_type              int64
gameweek                  int64
minutes                   int64
mins_score              float64
goals_score             float64
assists_score           float64
xg_score                float64
xgi_score               float64
cs_score                float64
bonus_score             float64
goals_conceded_score    float64
xgc_score               float64
composite_score         float64
dtype: object
